# FraudSentinel — Qwen3-14B Fine-Tuning Notebook
**Platform:** AMD MI300X (192 GB VRAM) via Unsloth  
**Dataset:** `naazimsnh02/fraud-financial-crime-qwen3-sft-v2` (11,816 ChatML conversations)  
**Task:** Supervised Fine-Tuning (SFT) for fraud detection & AML intelligence  

**What this notebook does:**
1. Installs all dependencies for ROCm/AMD
2. Prompts for HuggingFace token (required for model download & push)
3. Loads Qwen3-14B with LoRA adapters
4. Loads and prepares the FraudSentinel SFT dataset (ChatML format)
5. Trains with checkpointing so you can resume if interrupted
6. Runs a quick fraud-detection inference test
7. Pushes LoRA adapter + merged 16-bit model to HuggingFace Hub

**LoRA config:** r=16, alpha=32, all-linear targets (matches PRD spec)

In [3]:
%%bash
uv pip install -qU uv

ROCM_TAG="$({ command -v amd-smi >/dev/null 2>&1 && amd-smi version 2>/dev/null | awk -F'ROCm version: ' 'NF>1{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { [ -r /opt/rocm/.info/version ] && awk -F. '{print "rocm"$1"."$2; exit}' /opt/rocm/.info/version; } || { command -v hipconfig >/dev/null 2>&1 && hipconfig --version 2>/dev/null | awk -F': *' '/HIP version/{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { command -v dpkg-query >/dev/null 2>&1 && ver="$(dpkg-query -W -f='${Version}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; } || { command -v rpm >/dev/null 2>&1 && ver="$(rpm -q --qf '%{VERSION}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; })"
[ -n "$ROCM_TAG" ] || { echo "Could not detect ROCm. Install ROCm first or set ROCM_TAG manually."; exit 1; }
echo "Detected ROCm: $ROCM_TAG"
case "$ROCM_TAG" in
  rocm6.[0-4]|rocm7.[02]) T="$ROCM_TAG" ;;
  rocm6.*) T="rocm6.4" ;;
  *) T="rocm7.1" ;;
esac
pip install bitsandbytes
PYTORCH_INDEX_URL="https://download.pytorch.org/whl/${T}"
uv pip install --system -U --force-reinstall \
    torch torchvision torchaudio triton-rocm \
    --index-url "$PYTORCH_INDEX_URL"
uv pip install --system cut-cross-entropy torchao --no-deps
uv pip install --system -U --no-deps "unsloth[amd]" "unsloth_zoo[amd]"
uv pip install --system --no-deps -r "$(python -c 'import pathlib,site;print(next(p for r in [*site.getsitepackages(),site.getusersitepackages()] if (p:=pathlib.Path(r,"studio/backend/requirements/no-torch-runtime.txt")).exists()))')" torchao
uv pip install --system --no-deps -U "tokenizers>=0.22.0,<=0.23.0"

error: No virtual environment found; run `uv venv` to create an environment, or pass `--system` to install into a non-virtual environment


Detected ROCm: rocm7.0



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Using Python 3.12.11 environment at: /usr
Resolved 15 packages in 839ms
Prepared 15 packages in 1m 13s
Uninstalled 13 packages in 1.70s
Installed 15 packages in 484ms
 - filelock==3.20.0
 + filelock==3.29.0
 - fsspec==2025.9.0
 + fsspec==2026.4.0
 ~ jinja2==3.1.6
 - markupsafe==3.0.2
 + markupsafe==3.0.3
 ~ mpmath==1.3.0
 - networkx==3.5
 + networkx==3.6.1
 - numpy==2.2.6
 + numpy==2.4.4
 - pillow==11.3.0
 + pillow==12.2.0
 - setuptools==79.0.1
 + setuptools==70.2.0
 ~ sympy==1.14.0
 - torch==2.8.0+gitb2fb688 (from file:///install/torch-2.8.0%2Bgitb2fb688-cp312-cp312-linux_x86_64.whl)
 + torch==2.10.0+rocm7.0
 + torchaudio==2.10.0+rocm7.0
 - torchvision==0.23.0a0+824e8c8 (from file:///install/torchvision-0.23.0a0%2B824e8c8-cp312-cp312-linux_x86_64.whl)
 + torchvision==0.25.0+rocm7.0
 + triton-rocm==3.6.0
 ~ typing-extensions==4.15.0
Using Python 3.12.11 environme

In [4]:
# Additional packages needed for this notebook
!uv pip install --system -qqq sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer "transformers==4.56.2"
!uv pip install --system -qqq --no-deps accelerate peft "trl==0.22.2"

---
## Cell 2 — HuggingFace Authentication
Enter your HF token here. It will be used for:
- Downloading `Qwen/Qwen3-14B`
- Pushing the fine-tuned adapter + merged model to your HF Hub namespace

Get your token at: https://huggingface.co/settings/tokens  
Make sure it has **write** permissions.

In [ ]:
import os
from huggingface_hub import login, HfApi

# ── Configure these two values ──────────────────────────────────────────────
HF_TOKEN      = "your_hf_write_token"          # ← paste your HF write token
HF_USERNAME   = "your_hf_username"                  # ← your HF username / org
ADAPTER_REPO  = f"{HF_USERNAME}/fraudsentinel-qwen3-14b-lora"   # LoRA adapter
MERGED_REPO   = f"{HF_USERNAME}/fraudsentinel-qwen3-14b-merged" # Merged 16-bit
# ─────────────────────────────────────────────────────────────────────────────

login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()
print(f"Logged in as: {api.whoami()['name']}")
print(f"Adapter will be pushed to: {ADAPTER_REPO}")
print(f"Merged model will be pushed to: {MERGED_REPO}")

# Pre-create the repos so pushes don't fail on first run
for repo_id in [ADAPTER_REPO, MERGED_REPO]:
    try:
        api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, private=False)
        print(f"  ✅ Repo ready: {repo_id}")
    except Exception as e:
        print(f"  ⚠️  Could not create {repo_id}: {e}")

Logged in as: naazimsnh02
Adapter will be pushed to: naazimsnh02/fraudsentinel-qwen3-14b-lora
Merged model will be pushed to: naazimsnh02/fraudsentinel-qwen3-14b-merged
  ✅ Repo ready: naazimsnh02/fraudsentinel-qwen3-14b-lora
  ✅ Repo ready: naazimsnh02/fraudsentinel-qwen3-14b-merged


---
## Cell 3 — Load Qwen3-14B with LoRA
With 192 GB VRAM on the MI300X, we load in **bf16** (no quantization needed).  
This gives maximum accuracy and avoids any bitsandbytes CUDA/ROCm compatibility issues.

LoRA config matches the FraudSentinel PRD spec:  
`r=16, alpha=32, target_modules="all-linear"`

In [6]:
!uv pip install --system --force-reinstall /opt/rocm/share/amd_smi/

Using Python 3.12.11 environment at: /usr
Resolved 1 package in 2ms                                            
Prepared 1 package in 1.11s                                              
Uninstalled 1 package in 1ms
Installed 1 package in 2ms58ab (from file:///opt/rocm/share/
 - amdsmi==26.0.0+37d158ab (from file:///install/amdsmi-26.0.0%2B37d158ab-py3-none-any.whl)
 + amdsmi==26.0.0+37d158ab (from file:///opt/rocm/share/amd_smi)


In [2]:
import triton.language as tl

# Hot-fix the missing constexpr_function attribute for Triton 3.6.0+
if not hasattr(tl, 'constexpr_function'):
    tl.constexpr_function = lambda x: x

# Now import Unsloth safely
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096   
DTYPE = torch.bfloat16 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B", 
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = False, 
    load_in_8bit = False,
    full_finetuning = False,
    token = HF_TOKEN,
)

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 06-10 00:10:23 [__init__.py:224] Automatically detected platform rocm.
WARNING 06-10 00:10:23 [rocm.py:42] Failed to import from vllm._C with ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')
WARNING 06-10 00:10:23 [rocm.py:48] Failed to import from vllm._rocm_C with ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_rocm_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')
WARNING 06-10 00:10:23 [interface.py:178] Failed to import from vllm._C: ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00005-of-00006.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-14B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [3]:
# Add LoRA adapters — PRD spec: r=16, alpha=32, all-linear
model = FastLanguageModel.get_peft_model(
    model,
    r                        = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],  # PRD spec
    lora_alpha               = 32,
    lora_dropout             = 0,       # 0 is optimised by Unsloth
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",   # Unsloth saves ~30% VRAM
    random_state             = 3407,
    use_rslora               = False,
    loftq_config             = None,
)
model.print_trainable_parameters()

Unsloth 2026.6.1 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


trainable params: 64,225,280 || all params: 14,832,532,480 || trainable%: 0.4330


---
## Cell 4 — Load & Prepare FraudSentinel Dataset
The dataset is in **ChatML / HuggingFace messages format** (system / user / assistant).  
We apply the Qwen3 chat template with `enable_thinking=False` (fast-mode default per PRD).
Thinking mode is kept OFF during SFT — it is exposed as "Deep Analysis" at inference time only.

In [4]:
from datasets import load_dataset

print("Loading naazimsnh02/fraud-financial-crime-qwen3-sft-v2 ...")
ds = load_dataset(
    "naazimsnh02/fraud-financial-crime-qwen3-sft-v2",
    token=HF_TOKEN,
)
print(ds)
print("\nColumn names:", ds["train"].column_names)
print("\nFirst example (messages field):")
print(ds["train"][0]["messages"])

Loading naazimsnh02/fraud-financial-crime-qwen3-sft-v2 ...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/21.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11016 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/800 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'source', 'label', 'task'],
        num_rows: 11016
    })
    test: Dataset({
        features: ['messages', 'source', 'label', 'task'],
        num_rows: 800
    })
})

Column names: ['messages', 'source', 'label', 'task']

First example (messages field):
[{'role': 'system', 'content': 'You are FinCrime-Analyst, an AI assistant for an enterprise fraud-detection and financial-crime investigation platform. For each case you receive serialized transaction, behavioral, and contextual data. You must (1) assign a calibrated risk score from 0-100, (2) state a clear conclusion, (3) list the specific red flags you observed grounded in the data, (4) name the most likely fraud/laundering typology, and (5) recommend a concrete next action for the human investigator. Be precise, cite the evidence, and never invent facts that are not in the provided data. When asked for JSON, return a single valid JSON object with keys: risk_score (

In [5]:
# Apply Qwen3 chat template to every conversation.
# enable_thinking=False → no <think>...</think> CoT tokens during SFT.
# Investigators want concise structured JSON + prose, not CoT noise.

def apply_chat_template(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize           = False,
            add_generation_prompt = False,   # full turns, not open-ended
            enable_thinking    = False,       # fast-mode SFT
        )
        texts.append(text)
    return {"text": texts}

train_dataset = ds["train"].map(apply_chat_template, batched=True)
test_dataset  = ds["test"].map(apply_chat_template, batched=True)

print(f"\nTrain size : {len(train_dataset):,}")
print(f"Test  size : {len(test_dataset):,}")
print("\nSample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])

Map:   0%|          | 0/11016 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]


Train size : 11,016
Test  size : 800

Sample formatted text (first 500 chars):
<|im_start|>system
You are FinCrime-Analyst, an AI assistant for an enterprise fraud-detection and financial-crime investigation platform. For each case you receive serialized transaction, behavioral, and contextual data. You must (1) assign a calibrated risk score from 0-100, (2) state a clear conclusion, (3) list the specific red flags you observed grounded in the data, (4) name the most likely fraud/laundering typology, and (5) recommend a concrete next action for the human investigator. Be p


In [6]:
# Sanity-check token lengths so we understand padding needs
import numpy as np

sample_lengths = [
    len(tokenizer(ex["text"])["input_ids"])
    for ex in train_dataset.select(range(200))
]
print(f"Token length stats (sample of 200):")
print(f"  mean={np.mean(sample_lengths):.0f}, "
      f"p50={np.percentile(sample_lengths, 50):.0f}, "
      f"p90={np.percentile(sample_lengths, 90):.0f}, "
      f"p99={np.percentile(sample_lengths, 99):.0f}, "
      f"max={np.max(sample_lengths)}")
print(f"  MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} — sequences above this will be truncated")

Token length stats (sample of 200):
  mean=532, p50=538, p90=645, p99=729, max=751
  MAX_SEQ_LENGTH=4096 — sequences above this will be truncated


---
## Cell 5 — Configure Training with Checkpointing
Key settings for the MI300X run:
- `per_device_train_batch_size=2` (192 GB VRAM → big batch)
- `gradient_accumulation_steps=8` → effective batch = 16
- `save_strategy="steps"` + `save_steps=100` → checkpoint every 100 steps
- `resume_from_checkpoint=True` in `trainer.train()` → auto-resume if interrupted
- `num_train_epochs=2` (PRD recommendation for this dataset size)

In [9]:
from trl import SFTTrainer, SFTConfig

CHECKPOINT_DIR = "./fraudsentinel_checkpoints"
OUTPUT_DIR     = "./fraudsentinel_output"

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = train_dataset,
    eval_dataset   = test_dataset,
    args = SFTConfig(
        # ── Output & Checkpointing ─────────────────────────────────────────
        output_dir              = CHECKPOINT_DIR,
        save_strategy           = "steps",
        save_steps              = 100,          # checkpoint every 100 steps
        save_total_limit        = 5,            # keep last 5 checkpoints only
        load_best_model_at_end  = False,        # we want last, not best (SFT)
        # ── Evaluation ────────────────────────────────────────────────────
        eval_strategy           = "steps",
        eval_steps              = 100,
        # ── Data ──────────────────────────────────────────────────────────
        dataset_text_field      = "text",
        max_seq_length          = MAX_SEQ_LENGTH,
        packing                 = False,         
        # ── Batch Size & Accumulation ─────────────────────────────────────
        per_device_train_batch_size  = 2,       # MI300X 192 GB can handle this
        per_device_eval_batch_size   = 4,
        gradient_accumulation_steps  = 8,       # effective batch = 16
        # ── Training Schedule ─────────────────────────────────────────────
        num_train_epochs        = 2,            # PRD recommendation
        warmup_ratio            = 0.05,
        learning_rate           = 1e-4,         # PRD spec
        lr_scheduler_type       = "cosine",
        # ── Precision & Memory ────────────────────────────────────────────
        bf16                    = True,         # MI300X native bf16
        fp16                    = False,
        optim                   = "adamw_8bit", # saves memory even on 192 GB
        padding_free            = True,         # >17 GB VRAM — use flash attn packing
        # ── Logging ───────────────────────────────────────────────────────
        logging_steps           = 10,
        report_to               = "none",       # swap to "wandb" or "trackio" if needed
        # ── Misc ──────────────────────────────────────────────────────────
        weight_decay            = 0.001,
        seed                    = 3407,
        dataloader_num_workers  = 4,
    ),
)

print("Trainer configured.")
print(f"  Checkpoints → {CHECKPOINT_DIR}")
print(f"  Effective batch size: "
      f"{trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps}")

Trainer configured.
  Checkpoints → ./fraudsentinel_checkpoints
  Effective batch size: 16


---
## Cell 6 — Check GPU Memory Before Training

In [8]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU        : {gpu_stats.name}")
print(f"Total VRAM : {max_memory} GB")
print(f"Reserved   : {start_gpu_memory} GB  ({round(start_gpu_memory/max_memory*100,1)}%)")

GPU        : AMD Radeon Graphics
Total VRAM : 191.984 GB
Reserved   : 27.793 GB  (14.5%)


---
## Cell 7 — Train
Pass `resume_from_checkpoint=True` to auto-resume from the latest checkpoint in  
`fraudsentinel_checkpoints/` if this cell was interrupted and re-run.

In [10]:
import os

# Auto-detect whether a checkpoint exists to resume from
latest_checkpoint = None
if os.path.isdir(CHECKPOINT_DIR):
    checkpoints = sorted(
        [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[-1])
    )
    if checkpoints:
        latest_checkpoint = os.path.join(CHECKPOINT_DIR, checkpoints[-1])
        print(f"📂 Resuming from checkpoint: {latest_checkpoint}")
    else:
        print("🆕 No checkpoint found — starting fresh training run.")
else:
    print("🆕 No checkpoint dir found — starting fresh training run.")

trainer_stats = trainer.train(resume_from_checkpoint=latest_checkpoint)

🆕 No checkpoint found — starting fresh training run.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,016 | Num Epochs = 2 | Total steps = 1,378
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 64,225,280 of 14,832,532,480 (0.43% trained)


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/dist-packages/unsloth/kernels/utils.py:1079: UserWarning: Unmatched validator: "HIP_VERSION" is required, but the tuning results does not provide it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:411.)
  out = torch_matmul(X, W.t(), out = out)
/usr/local/lib/python3.12/dist-packages/unsloth/kernels/utils.py:1079: UserWarning: Unmatched validator: "ROCM_VERSION" is provided, but pytorch is unable to consume it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:419.)
  out = torch_matmul(X, W.t(), out = out)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.220100,0.222961
200,0.197000,0.201034
300,0.190400,0.192675
400,0.186800,0.189027
500,0.183800,0.184117
600,0.177600,0.180371
700,0.173700,0.177213
800,0.167000,0.173893
900,0.167300,0.170731
1000,0.162000,0.168381


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [11]:
# Final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"Training time : {trainer_stats.metrics['train_runtime']:.0f}s "
      f"({round(trainer_stats.metrics['train_runtime']/60, 1)} min)")
print(f"Train loss    : {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"Peak VRAM     : {used_memory} GB  ({used_percentage}% of {max_memory} GB)")
print(f"LoRA overhead : {used_memory_for_lora} GB  ({lora_percentage}% of max)")

Training time : 4230s (70.5 min)
Train loss    : 0.24665129721078194
Peak VRAM     : 39.838 GB  (20.751% of 191.984 GB)
LoRA overhead : 12.045 GB  (6.274% of max)


---
## Cell 8 — Inference Test (Fraud Detection)
Validate the fine-tuned model with a representative FraudSentinel prompt before pushing.
Tests both card fraud (structured JSON) and AML (analyst prose) modes.

In [13]:
from transformers import TextStreamer

def run_inference(messages, enable_thinking=False, max_new_tokens=512):
    """Run inference with the fine-tuned model."""
    text = tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = True,
        enable_thinking       = enable_thinking,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True)
    outputs = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature    = 0.7,
        top_p          = 0.8,
        top_k          = 20,
        streamer       = streamer,
    )
    return outputs


# ── Test 1: Structured JSON risk scoring (card fraud) ────────────────────────
print("=" * 70)
print("TEST 1: Structured JSON risk scoring — card fraud")
print("=" * 70)

card_fraud_messages = [
    {
        "role": "system",
        "content": (
            "You are FraudSentinel, an enterprise fraud detection assistant. "
            "Analyse transactions and return structured risk assessments. "
            "When asked for structured output, respond with valid JSON only."
        )
    },
    {
        "role": "user",
        "content": (
            "Analyse this card transaction and return a structured JSON risk assessment.\n\n"
            "Transaction:\n"
            "  amount: $1,247.89\n"
            "  merchant: CRYPTO_EXCHANGE_XYZ\n"
            "  category: misc_net\n"
            "  timestamp: 2024-03-15 03:22:41\n"
            "  card_id: xxxx-xxxx-xxxx-4821\n"
            "  cardholder_zip: 90210\n"
            "  merchant_lat: 37.77, merchant_long: -122.41\n"
            "  cardholder_lat: 34.05, cardholder_long: -118.24\n"
            "  transactions_last_24h: 7\n"
            "  spend_last_24h: $3,891.20\n"
            "  category_p95_amount: $383.21\n\n"
            "Tier-1 score: 0.94 (CRITICAL)\n"
            "Signals fired: amount_exceeds_category_p95, unusual_hour_activity, "
            "high_velocity_24h, high_risk_merchant_category"
        )
    }
]

_ = run_inference(card_fraud_messages, enable_thinking=False, max_new_tokens=600)

TEST 1: Structured JSON risk scoring — card fraud
{
  "risk_level": "CRITICAL",
  "score": 0.94,
  "primary_feature": "amount_exceeds_category_p95",
  "secondary_features": [
    "unusual_hour_activity",
    "high_velocity_24h",
    "high_risk_merchant_category"
  ],
  "explanation": "Transaction amount $1,247.89 exceeds the 95th-percentile ($383.21) for 'misc_net' purchases; Transaction occurred at an unusual hour (03:22 local); High-velocity activity: 7 transactions on this card within 24h, totaling $3,891.20.",
  "recommended_action": "BLOCK/STEP_UP_AUTH"
}<|im_end|>


In [14]:
# ── Test 2: AML analyst prose ────────────────────────────────────────────────
print("\n" + "=" * 70)
print("TEST 2: AML analyst prose explanation")
print("=" * 70)

aml_messages = [
    {
        "role": "system",
        "content": (
            "You are FraudSentinel, an AML (anti-money laundering) investigation assistant. "
            "Provide clear, evidence-grounded explanations for investigators. "
            "Cite specific transaction features. Recommend concrete next actions."
        )
    },
    {
        "role": "user",
        "content": (
            "Explain the AML risk for this inter-bank transfer cluster.\n\n"
            "Account A001 → A002: $9,800 USD (ACH)\n"
            "Account A001 → A003: $9,750 USD (ACH)\n"
            "Account A001 → A004: $9,700 USD (ACH)\n"
            "All three transfers occurred within 47 minutes on 2024-03-14.\n"
            "A001 out-degree: 18 in last 30 days. All amounts just under $10,000.\n"
            "A002, A003, A004 received no other inbound transfers in past 90 days.\n\n"
            "Tier-1 GNN score: 0.91. Signals: structuring_pattern, fan_out, "
            "round_number_threshold_avoidance"
        )
    }
]

_ = run_inference(aml_messages, enable_thinking=False, max_new_tokens=600)


TEST 2: AML analyst prose explanation
Risk level: HIGH (0.91/1.0). This is classic structuring: the same account (A001) sends three transfers within 47 minutes, each just below $10,000, to beneficiaries (A002, A003, A004) that receive no other fund flow in the 90-day window. The pattern indicates a single source funneling funds to multiple destinations, consistent with placement/layering. The fan-out (one-to-many) and round-number threshold avoidance are red flags.

Recommended next action: [STEP_UP_AUTH] Hold the account on monitor; trace the beneficiary accounts; prepare a draft Suspicious Activity Report (SAR) documenting the structuring pattern.<|im_end|>


In [15]:
# ── Test 3: Deep Analysis mode (thinking ON) ─────────────────────────────────
print("\n" + "=" * 70)
print("TEST 3: Deep Analysis mode (thinking=True) — complex case")
print("=" * 70)

deep_analysis_messages = [
    {
        "role": "system",
        "content": (
            "You are FraudSentinel in Deep Analysis mode. "
            "Think step-by-step through the evidence before rendering a verdict. "
            "Consider alternative explanations and assess confidence carefully."
        )
    },
    {
        "role": "user",
        "content": (
            "Deep analysis requested. Should we file a SAR for this case?\n\n"
            "Entity: IMPORT_EXPORT_LLC\n"
            "Pattern: 23 wire transfers over 60 days, alternating USD↔EUR↔HKD\n"
            "Total: $2.3M outbound, $2.1M inbound\n"
            "Counterparties span 7 jurisdictions including 2 FATF grey-list countries\n"
            "Declared business: import/export of electronics\n"
            "Account age: 14 months\n"
            "Prior SARs on counterparties: 2\n"
        )
    }
]

_ = run_inference(deep_analysis_messages, enable_thinking=True, max_new_tokens=1024)


TEST 3: Deep Analysis mode (thinking=True) — complex case
<think>

</think>

Activity pattern: 23 wire transfers over 60 days, alternating USD↔EUR↔HKD
Red flags:
- Rapid movement across 3 currencies and 7 jurisdictions
- 23 transfers in 60 days (4.6 per 30 days)
- Material fund flow asymmetry: $2.3M out / $2.1M in
- Counterparties span 7 jurisdictions including 2 FATF grey-list countries
- Declared import/export business does not justify the cross-border fund flow intensity

Conclusion: FRAUDULENT / SUSPICIOUS
Recommended action: FILE a Tier 1 SAR and hold the account pending investigation.
Confidence: 91/100<|im_end|>


---
## Cell 9 — Save & Push to HuggingFace Hub
Three things are pushed:
1. **LoRA adapter** (lightweight, ~100MB) — for vLLM hot-swapping per tenant
2. **Merged 16-bit model** — for direct vLLM deployment without adapter infrastructure
3. **Tokenizer** — alongside both repos

In [16]:
import os

# ── Step 1: Save LoRA adapter locally ────────────────────────────────────────
LOCAL_ADAPTER_PATH = "./fraudsentinel_lora"
LOCAL_MERGED_PATH  = "./fraudsentinel_merged_16bit"

print("Saving LoRA adapter locally ...")
model.save_pretrained(LOCAL_ADAPTER_PATH)
tokenizer.save_pretrained(LOCAL_ADAPTER_PATH)
print(f"  ✅ Saved to {LOCAL_ADAPTER_PATH}")

# ── Step 2: Push LoRA adapter to HF Hub ──────────────────────────────────────
print(f"\nPushing LoRA adapter to {ADAPTER_REPO} ...")
model.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
print(f"  ✅ Adapter live at: https://huggingface.co/{ADAPTER_REPO}")

Saving LoRA adapter locally ...
  ✅ Saved to ./fraudsentinel_lora

Pushing LoRA adapter to naazimsnh02/fraudsentinel-qwen3-14b-lora ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/naazimsnh02/fraudsentinel-qwen3-14b-lora


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  ✅ Adapter live at: https://huggingface.co/naazimsnh02/fraudsentinel-qwen3-14b-lora


In [17]:
# ── Step 3: Save & push merged 16-bit model (for vLLM deployment) ────────────
# This merges LoRA weights back into base model — needed for direct vLLM loading
# without adapter infrastructure. Larger (~28GB) but simplest deployment path.

print(f"Saving & pushing merged 16-bit model to {MERGED_REPO} ...")
print("  This may take 10-20 minutes (serialising 14B parameters) ...")
model.push_to_hub_merged(
    MERGED_REPO,
    tokenizer,
    save_method = "merged_16bit",
    token       = HF_TOKEN,
)
print(f"  ✅ Merged model live at: https://huggingface.co/{MERGED_REPO}")

Saving & pushing merged 16-bit model to naazimsnh02/fraudsentinel-qwen3-14b-merged ...
  This may take 10-20 minutes (serialising 14B parameters) ...
Unsloth: Saving to naazimsnh02/fraudsentinel-qwen3-14b-merged will fail, but using a temp folder works! Switching to a temp folder then uploading!


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:  83%|████████▎ | 5/6 [04:07<00:49, 49.50s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 6/6 [04:51<00:00, 48.51s/it]


Unsloth: Merge process complete. Saved to `/tmp/tmp3b9xx1z4`
  ✅ Merged model live at: https://huggingface.co/naazimsnh02/fraudsentinel-qwen3-14b-merged


In [18]:
# ── Step 4: Create model card ─────────────────────────────────────────────────
model_card_content = f"""---
language:
- en
license: apache-2.0
base_model: Qwen/Qwen3-14B
tags:
- fraud-detection
- anti-money-laundering
- financial-crime
- lora
- unsloth
- qwen3
datasets:
- naazimsnh02/fraud-financial-crime-qwen3-sft-v2
---

# FraudSentinel — Qwen3-14B LoRA (Tier-2 Intelligence Layer)

Fine-tuned Qwen3-14B for enterprise fraud detection and AML investigation.
Part of the [FraudSentinel](https://huggingface.co/{HF_USERNAME}) two-tier platform.

## Capabilities
- Structured JSON risk scoring (0.0–1.0, LOW/MEDIUM/HIGH/CRITICAL)
- Explainable alerts with evidence-grounded red flags
- Typology classification (card fraud + AML patterns)
- 6-level investigator action recommendations
- SAR (Suspicious Activity Report) drafting support
- Multi-turn HITL investigator dialogue
- Deep Analysis mode (Chain-of-Thought via Qwen3 thinking tokens)

## Training Details
- **Base model:** Qwen/Qwen3-14B (Apache-2.0)
- **Method:** SFT + LoRA (r=16, alpha=32, all-linear)
- **Dataset:** `naazimsnh02/fraud-financial-crime-qwen3-sft-v2` (11,816 conversations)
- **Hardware:** AMD MI300X 192 GB VRAM
- **Framework:** Unsloth + TRL

## Usage
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained("{ADAPTER_REPO}")
```

## Limitations
Prototype for research. Do not use for real customer adjudication without
independent validation, bias review, and human-in-the-loop controls.
"""

# Save model card to both local dirs and push
for local_path in [LOCAL_ADAPTER_PATH]:
    with open(os.path.join(local_path, "README.md"), "w") as f:
        f.write(model_card_content)

api.upload_file(
    path_or_fileobj=model_card_content.encode(),
    path_in_repo="README.md",
    repo_id=ADAPTER_REPO,
    token=HF_TOKEN,
)
print("✅ Model card uploaded.")

✅ Model card uploaded.


---
## Cell 10 — Summary & Next Steps

print("=" * 70)
print("  FraudSentinel Qwen3-14B — Training Complete")
print("=" * 70)
print(f"\n  Training time : {trainer_stats.metrics['train_runtime']/60:.1f} min")
print(f"  Peak VRAM     : {used_memory} GB / {max_memory} GB")
print(f"\n  Artefacts pushed:")
print(f"    LoRA adapter  → https://huggingface.co/{ADAPTER_REPO}")
print(f"    Merged 16-bit → https://huggingface.co/{MERGED_REPO}")
print(f"\n  Local checkpoints: {CHECKPOINT_DIR}/")
print(f"  Local LoRA dir:    {LOCAL_ADAPTER_PATH}/")
print(f"  Local merged dir:  {LOCAL_MERGED_PATH}/")
print("""
  Next steps (FraudSentinel roadmap):
    1. Wrap merged model in FastAPI at POST /v1/analyze  (Task #2 in PRD)
    2. Deploy via vLLM:
         vllm serve {MERGED_REPO} --gpu-memory-utilization 0.85
    3. For multi-tenant LoRA swapping, load the adapter repo into vLLM:
         vllm serve Qwen/Qwen3-14B --enable-lora
           --lora-modules fraudsentinel={ADAPTER_REPO}
    4. Wire Tier-1 scores into the /v1/analyze request body
    5. Build React dashboard (Task #3 in PRD)
""")